In [ ]:
import numpy as np
import pandas as pd
import rasterio as rio
from tqdm import tqdm
from pathlib import Path

In [ ]:
ROOT = Path.cwd().parent
DATA = ROOT / 'data'
RESULTS = ROOT / 'results'
(RESULTS / 'biodiversity-risk').mkdir(parents=True, exist_ok=True)

# 6 — Biodiversity risk assessment (per CE scenario × β)

Overlays each projected mining footprint (`mining-area-expansion-2050-…`, from `3-mining-area-simulation/6-estimate-mining-area`) with rarity-weighted richness (RWR) to estimate the **mining-biodiversity overlap**: the fraction of each cell that is mined times the rarity-weighted richness it holds, summed globally and broken down by taxon and IUCN Red-List category.

This is a **risk / exposure** metric (potential range loss from projected expansion), not an observed impact.

## Cross (M/M central case)
scenario × β sweep — read footprints from `mining-area/`, write overlap outputs to `biodiversity-risk/`.

$$\text{mining-biodiversity overlap} = \sum_{\text{cells}} \frac{\text{footprint (km}^2)}{\text{cell area}} \times \text{RWR}$$

## Inputs
- footprint: `mining-area/mining-area-expansion-2050-{scenario}-B{beta}-{res}-D{Ld}-R{Lr}-A{La}.tif` (carries all three uncertainty dimensions; overlap output mirrors the same tag)
- RWR (overall + per taxon × IUCN category): `area-of-habitat-{res}/rarity-weighted-richness-*-{res}.tif`

In [ ]:
def assess_biodiversity_risk(scenario, beta, demand_level='M', reserve_level='M', area_level='M', res='50km'):
    """mining-biodiversity overlap = (footprint / cell_area) * RWR, summed; broken down by
    taxon x IUCN category. A risk/exposure metric, not an observed impact.

    `tag` carries all three uncertainty dimensions, matching the footprint filename.
    DM-RM-AM is central; bounds are selected from the complete 27-case ensemble."""
    scale = {'10km': 10, '50km': 50}[res]
    tag = f'{scenario}-B{beta:.2f}-{res}-D{demand_level}-R{reserve_level}-A{area_level}'
    AOH = RESULTS / f'area-of-habitat-{res}'

    country_raster_ = rio.open(RESULTS / 'country-rasters' / f'country-raster-eck4-{res}.tif')
    footprint = rio.open(RESULTS / 'mining-area' / f'mining-area-expansion-2050-{tag}.tif').read(1)
    if not np.isfinite(footprint).all() or (footprint < 0).any():
        raise ValueError(f'Invalid footprint for {tag}; regenerate it with 6-estimate-mining-area before assessing risk')
    RWR = rio.open(AOH / f'rarity-weighted-richness-vertebrates-{res}.tif').read(1)
    overlap = (footprint / scale ** 2) * RWR

    taxa = ['mammals', 'birds', 'amphibians', 'reptiles', 'vertebrates']
    df = pd.DataFrame(index=['CR', 'EN', 'VU', 'NT', 'LC/DD', 'total'], columns=taxa, dtype=float)
    for taxon in taxa:
        for category in ['CR', 'EN', 'VU', 'NT', 'total']:
            suffix = '' if category == 'total' else f'-{category}'
            RWR_t = rio.open(AOH / f'rarity-weighted-richness-{taxon}{suffix}-{res}.tif').read(1)
            df.loc[category, taxon] = float(np.sum((footprint / scale ** 2) * RWR_t))
    df.loc['LC/DD'] = df.loc['total'] - df.loc[['CR', 'EN', 'VU', 'NT']].sum()
    df = df.loc[['LC/DD', 'NT', 'VU', 'EN', 'CR'], ['mammals', 'birds', 'amphibians', 'reptiles']]

    out_tif = RESULTS / 'biodiversity-risk' / f'mining-biodiversity-overlap-2050-{tag}.tif'
    with rio.open(out_tif, 'w', driver='GTiff',
        height=country_raster_.shape[0], width=country_raster_.shape[1], count=1,
        dtype=overlap.dtype, crs=country_raster_.crs,
        transform=country_raster_.transform, compress='lzw') as dst:
        dst.write(overlap, 1)
    df.to_csv(RESULTS / 'biodiversity-risk' / f'mining-biodiversity-overlap-2050-{tag}.csv')

    print(f'  {tag}: mining-biodiversity overlap = {float(np.sum(overlap)):.4g}')
    return overlap, df

## Run — scenario × β cross (30, M/M)

In [ ]:
import json
_cfg = json.load(open(RESULTS / 'sweep-config.json'))   # sweep grid defined once in 5-2-map-extraction
SCENARIOS, BETAS = _cfg['SCENARIOS'], _cfg['BETAS']

records = []
for scen in tqdm(SCENARIOS):
    for b in BETAS:
        overlap, _ = assess_biodiversity_risk(scen, b, demand_level='M', reserve_level='M')
        records.append({'scenario': scen, 'beta': b, 'overlap': float(np.sum(overlap))})

risk_df = pd.DataFrame(records)
risk_df.to_csv(RESULTS / 'biodiversity-risk' / 'biodiversity-risk-summary-DM-RM.csv', index=False)
risk_df.round(3)

## Run — `scenario_set_1` × β0 × full 3×3×3 uncertainty factorial (216 overlaps)

Converts every footprint from the β0 demand×reserve×area factorial in `3-mining-area-simulation/6-estimate-mining-area` into mining-biodiversity overlap. Each call also writes the per-combo overlap raster + taxon×IUCN CSV; the tidy summary lands in `biodiversity-risk-summary-B0-DRA-factorial.csv` (216 rows).

→ `biodiversity-risk/mining-biodiversity-overlap-2050-{scen}-B0.00-50km-D{Ld}-R{Lr}-A{La}.tif` (+ taxon CSV)

In [ ]:
scenario_set_1 = ['BAU', 'MIU', 'LE', 'RE', 'RC', 'SS', 'CCR', 'CE']
LEVELS = ['L', 'M', 'H']

# Convert every footprint from 6-estimate-mining-area's beta0 factorial into mining-biodiversity
# overlap. Each call also writes the per-combo overlap raster + taxon x IUCN CSV.
# 8 scenarios x {D,R,A} in {L,M,H}^3 = 216 overlaps, all at beta = 0.
records = []
for scen in tqdm(scenario_set_1):
    for dl in LEVELS:
        for rl in LEVELS:
            for al in LEVELS:
                overlap, _ = assess_biodiversity_risk(scen, 0.0, demand_level=dl,
                                                      reserve_level=rl, area_level=al)
                records.append({'scenario': scen, 'beta': 0.0, 'demand': dl,
                                'reserve': rl, 'area': al, 'overlap': float(np.sum(overlap))})

factorial_df = pd.DataFrame(records)
factorial_df.to_csv(RESULTS / 'biodiversity-risk' / 'biodiversity-risk-summary-B0-DRA-factorial.csv',
                    index=False)
factorial_df.round(3)

## Uncertainty bounds from all 27 combinations at each signpost

Compute all D/R/A outcomes, then select their minimum and maximum globally. Taxon and regional bounds must be computed independently from their own 27 totals.


In [ ]:
SIGNPOSTS = [('BAU', 0.0), ('BAU', 0.7), ('CE', 0.0), ('CE', 0.7)]
rows = []
for scen, b in SIGNPOSTS:
    outcomes = []
    for dl in ['L', 'M', 'H']:
        for rl in ['L', 'M', 'H']:
            for al in ['L', 'M', 'H']:
                overlap, _ = assess_biodiversity_risk(scen, b, demand_level=dl, reserve_level=rl, area_level=al)
                outcomes.append(dict(scenario=scen, beta=b, demand=dl, reserve=rl, area=al, overlap=float(np.sum(overlap))))
    rows.extend([dict(min(outcomes, key=lambda x:x['overlap']), bound='lower'),
                 dict(max(outcomes, key=lambda x:x['overlap']), bound='upper')])
bounds_df = pd.DataFrame(rows)
bounds_df.to_csv(RESULTS / 'biodiversity-risk' / 'biodiversity-risk-bounds-signposts.csv', index=False)
bounds_df.round(4)
